# Predicting the video

Requirements: 
tensorflow==2.3.0
keras==2.4.3
scikit-learn==0.23.2


In [ ]:
cv2.__version__

'4.5.1'

In [1]:
from keras.models import load_model
from collections import deque
import matplotlib.pyplot as plt
import numpy as np
import argparse
import pickle
import cv2

In [ ]:
# WORKING PROJECT (MobileNetV2 weights)
import numpy as np
import argparse
import pickle
import cv2
import os
import time 
from collections import deque

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

def build_trained_model(weights_path):
        base_model = MobileNetV2(weights=None, include_top=False, input_shape=(224, 224, 3))
        x = base_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dense(256, activation='relu')(x)
        x = Dropout(0.3)(x)
        output = Dense(1, activation='sigmoid')(x)
        model = Model(inputs=base_model.input, outputs=output)
        model.load_weights(weights_path)
        return model

def print_results(video, limit=None):
        if not os.path.exists('output'):
            os.mkdir('output')

        print("Loading model ...")
        model = build_trained_model(r"C:\Users\shobh\Downloads\Real-life-violence-detection-main\Real-life-violence-detection-main\mobilenetv2_violence.weights.h5")
        Q = deque(maxlen=128)
        vs = cv2.VideoCapture(video)
        writer = None
        (W, H) = (None, None)
        count = 0     
        while True:
            (grabbed, frame) = vs.read()
            if not grabbed:
                break
            if W is None or H is None:
                (H, W) = frame.shape[:2]
            output = frame.copy()
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224)).astype("float32")
            frame = frame.reshape(224, 224, 3) / 255.0
            preds = model.predict(np.expand_dims(frame, axis=0))[0]
            Q.append(preds)
            results = np.array(Q).mean(axis=0)
            i = (preds > 0.50)[0]
            label = i
            text_color = (0, 255, 0)
            if label:
                text_color = (0, 0, 255)
            text = "Violence: {}".format(label)
            FONT = cv2.FONT_HERSHEY_SIMPLEX 
            cv2.putText(output, text, (35, 50), FONT,1.25, text_color, 3)
            if writer is None:
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter("output/v_output.mp4", fourcc, 30,(W, H), True)
            writer.write(output)
            cv2.imshow("Output", output)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
        print("[INFO] cleaning up...")
        if writer is not None:
            writer.release()
        vs.release()

In [ ]:
V_path = r"C:\Users\shobh\Downloads\Real-life-violence-detection-main\Real-life-violence-detection-main\4121252-uhd_3840_2160_25fps.mp4"
NV_path = None

In [4]:
print_results(V_path)

Loading model ...
[INFO] cleaning up...


# END